# Cross-Asset Comparison
Compare all tracked symbols side-by-side:
correlation matrix, relative performance, volume dominance, vol ranking.

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv()

EXCHANGE = 'binance'
DAYS     = 30

ch = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', 'clickhouse'),
    port=int(os.getenv('CLICKHOUSE_HTTP_PORT', '8123')),
    database=os.getenv('CLICKHOUSE_DB', 'crypto'),
    username=os.getenv('CLICKHOUSE_USER', 'crypto_user'),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
)
print(f'Exchange: {EXCHANGE} | Last {DAYS} days')

## 1. Relative Performance (rebased to 100)

In [ ]:
daily = ch.query_df(f"""
SELECT
    symbol,
    toDate(open_time)        AS trade_date,
    argMax(close, open_time) AS day_close
FROM crypto.fact_candles
WHERE exchange = '{EXCHANGE}'
  AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
GROUP BY symbol, trade_date
ORDER BY symbol, trade_date
""")

# Rebase each symbol to 100 at first observation
daily['rebased'] = daily.groupby('symbol')['day_close'].transform(
    lambda x: x / x.iloc[0] * 100
)

fig = px.line(daily, x='trade_date', y='rebased', color='symbol',
              title=f'Relative Performance — rebased to 100 (last {DAYS} days)',
              labels={'rebased': 'Indexed (100 = start)', 'trade_date': 'Date'})
fig.add_hline(y=100, line_dash='dash', line_color='grey')
fig.show()

## 2. Return Correlation Matrix

In [ ]:
# Pivot to wide format: date x symbol
pivot = daily.pivot(index='trade_date', columns='symbol', values='day_close')
log_returns = np.log(pivot / pivot.shift(1)).dropna()
corr = log_returns.corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
                zmin=-1, zmax=1,
                title=f'Daily Log-Return Correlation (last {DAYS} days)')
fig.show()

## 3. Volume Dominance (share of total USDT volume)

In [ ]:
vol_share = ch.query_df(f"""
SELECT
    symbol,
    round(sum(quote_volume) / 1e6, 1)          AS total_vol_M,
    round(sum(quote_volume) * 100.0
          / sum(sum(quote_volume)) OVER (), 1)  AS vol_share_pct
FROM crypto.fact_candles
WHERE exchange = '{EXCHANGE}'
  AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
GROUP BY symbol
ORDER BY total_vol_M DESC
""")

fig = px.pie(vol_share, names='symbol', values='total_vol_M',
             title=f'USDT Volume Share (last {DAYS} days)',
             hover_data=['vol_share_pct'])
fig.show()

## 4. Volatility vs Return Scatter

In [ ]:
summary = ch.query_df(f"""
WITH perf AS (
    SELECT symbol,
        round((argMax(day_close, trade_date) - argMin(day_close, trade_date))
              / nullIf(argMin(day_close, trade_date), 0) * 100, 1) AS return_30d
    FROM (
        SELECT symbol, toDate(open_time) AS trade_date,
               argMax(close, open_time) AS day_close
        FROM crypto.fact_candles
        WHERE exchange='{EXCHANGE}' AND interval='1m'
          AND open_time >= now() - INTERVAL {DAYS} DAY
        GROUP BY symbol, trade_date
    )
    GROUP BY symbol
)
SELECT v.symbol,
    round(v.realized_vol_30d * 100, 1) AS vol_30d_pct,
    p.return_30d
FROM crypto.mart_volatility v
JOIN perf p USING (symbol)
WHERE (v.exchange, v.symbol, v.window_start) IN (
    SELECT exchange, symbol, max(window_start)
    FROM crypto.mart_volatility GROUP BY exchange, symbol
)
""")

fig = px.scatter(summary, x='vol_30d_pct', y='return_30d', text='symbol',
                 title=f'Risk vs Return (last {DAYS} days)',
                 labels={'vol_30d_pct': 'Realized Vol 30d (%)',
                         'return_30d': 'Return 30d (%)'},
                 color='return_30d',
                 color_continuous_scale='RdYlGn',
                 color_continuous_midpoint=0)
fig.update_traces(textposition='top center', marker_size=12)
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.show()

## 5. Bull/Bear Candle Ratio by Symbol

In [ ]:
candle_stats = ch.query_df(f"""
SELECT
    symbol,
    count()                                       AS total,
    round(sum(is_bullish) * 100.0 / count(), 1)   AS pct_bull,
    round((1 - sum(is_bullish) * 1.0 / count()) * 100, 1) AS pct_bear
FROM crypto.fact_candles
WHERE exchange = '{EXCHANGE}' AND interval = '1m'
  AND open_time >= now() - INTERVAL {DAYS} DAY
GROUP BY symbol
ORDER BY pct_bull DESC
""")

fig = go.Figure(data=[
    go.Bar(name='Bullish %', x=candle_stats['symbol'], y=candle_stats['pct_bull'],
           marker_color='mediumseagreen'),
    go.Bar(name='Bearish %', x=candle_stats['symbol'], y=candle_stats['pct_bear'],
           marker_color='salmon'),
])
fig.update_layout(barmode='stack', title=f'Bull/Bear Candle Ratio (last {DAYS} days, 1m)',
                  yaxis_title='%', xaxis_title='Symbol')
fig.add_hline(y=50, line_dash='dash', line_color='grey')
fig.show()